# 웨이퍼 결함 — 탐지 + 분류 통합 학습 (기획서 F-02 / F-04 대응)

이 노트북 한 개로 **두 모델**을 학습·저장합니다.
1. **객체 탐지 (YOLOv8n)** — 단일 결함 패턴 이미지에서 결함 영역을 박스로 탐지 (F-02: 빨간 네모 박스 시각화)
2. **다중라벨 분류 (Swin Transformer)** — 혼합 패턴까지 8종 결함을 동시 예측 (F-04 리포트의 근거)

> 데이터: MixedWM38 (`Wafer_Map_Datasets.npz`) — `arr_0`(N,52,52) 0/1/2, `arr_1`(N,8) 멀티핫
> 결과물: `runs/detect/.../best.pt`, `swin_multilabel.pt`, `label_map.json` → Streamlit 데모가 로드
> Colab: 런타임 → GPU(T4) 권장.

## 0. 설치 & 임포트

In [ ]:
!pip -q install ultralytics timm torch torchvision scikit-learn matplotlib scipy tqdm
import numpy as np, os, json, glob
import matplotlib.pyplot as plt
from pathlib import Path
print('done')

## 1. 데이터 로드
`Wafer_Map_Datasets.npz`를 Google Drive나 업로드 경로에서 로드합니다. (GitHub: Junliangwangdhu/WaferMap)

In [ ]:
# 경로만 본인 환경에 맞게 수정
NPZ_PATH = 'Wafer_Map_Datasets.npz'   # 예: '/content/drive/MyDrive/superb-spot/Wafer_Map_Datasets.npz'
data = np.load(NPZ_PATH)
X = data['arr_0']            # (N,52,52) 값 0/1/2
Y = data['arr_1'].astype(int)# (N,8) 멀티핫
print('X', X.shape, 'Y', Y.shape)

LABELS = ['Center','Donut','Edge_Loc','Edge_Ring','Loc','Near_Full','Scratch','Random']
json.dump({'labels':LABELS}, open('label_map.json','w'), ensure_ascii=False, indent=2)

# 클래스 분포
import collections
cnt = Y.sum(axis=0)
for k,l in enumerate(LABELS): print(f'{l:10s}: {int(cnt[k])}')
print('결함 없는(정상) 샘플:', int((Y.sum(axis=1)==0).sum()))
print('단일 패턴 샘플:', int((Y.sum(axis=1)==1).sum()))
print('혼합 패턴 샘플:', int((Y.sum(axis=1)>=2).sum()))

## 2. 시각화 (샘플 몇 개)
0=빈공간(검정), 1=정상 die(회색), 2=불량 die(빨강)

In [ ]:
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#0b1021','#94a3b8','#ef4444'])
idxs = np.random.RandomState(0).choice(len(X), 8, replace=False)
plt.figure(figsize=(14,4))
for j,i in enumerate(idxs):
    plt.subplot(2,4,j+1)
    plt.imshow(X[i], cmap=cmap, vmin=0, vmax=2)
    act = [LABELS[k] for k in range(8) if Y[i,k]==1] or ['Normal']
    plt.title('+'.join(act), fontsize=8); plt.axis('off')
plt.tight_layout(); plt.show()

## 3. 객체 탐지용 YOLO 데이터셋 생성 (핵심)
**정직한 라벨 전략**: 혼합 이미지는 패턴별 위치 GT가 없으므로, **단일 패턴 이미지**만 사용해
결함 픽셀(값 2)의 바운딩 박스를 자동 생성 → 그 이미지의 단일 클래스로 라벨링.
- 국소 패턴(Center/Loc/Edge*)은 최대 연결요소 박스, Scratch는 선형 박스, Near_Full/Random은 전체 결함 영역 박스.
- 결과: `wafer_yolo/images/{train,val}`, `labels/{train,val}` + `data.yaml`

In [ ]:
from scipy import ndimage
from PIL import Image
from sklearn.model_selection import train_test_split

IMG = 256                      # YOLO 입력 크기 (52 → 256 업스케일)
S = IMG / 52.0
root = Path('wafer_yolo');
for sub in ['images/train','images/val','labels/train','labels/val']:
    (root/sub).mkdir(parents=True, exist_ok=True)

def to_rgb(a):
    # 0 검정, 1 회색, 2 빨강
    rgb = np.zeros((*a.shape,3), np.uint8)
    rgb[a==1] = (148,163,184); rgb[a==2] = (239,68,68)
    im = Image.fromarray(rgb).resize((IMG,IMG), Image.NEAREST)
    return im

def boxes_for(a, cls):
    """단일 패턴 결함(값2) 픽셀 → YOLO 박스 리스트 [(cls,cx,cy,w,h)] (정규화)"""
    mask = (a==2)
    if mask.sum()==0: return []
    lbl, n = ndimage.label(mask)
    comps = []
    for c in range(1, n+1):
        ys,xs = np.where(lbl==c)
        if len(xs) < 3:   # 노이즈 제거
            continue
        comps.append((xs.min(),ys.min(),xs.max(),ys.max(),len(xs)))
    if not comps: return []
    name = LABELS[cls]
    if name in ('Near_Full','Random'):
        # 흩어진 패턴 → 전체 결함 영역 하나의 박스
        ys,xs = np.where(mask)
        comps = [(xs.min(),ys.min(),xs.max(),ys.max(),len(xs))]
    elif name == 'Scratch':
        comps = [max(comps, key=lambda c:c[4])]  # 가장 긴 선형 요소
    else:
        comps = [max(comps, key=lambda c:c[4])]  # 국소 → 최대 요소
    out = []
    for x0,y0,x1,y1,_ in comps:
        # 52 grid → 정규화 (0~1). 픽셀 경계 보정 +1
        cx = ((x0+x1+1)/2)/52.0; cy = ((y0+y1+1)/2)/52.0
        w  = (x1-x0+1)/52.0;      h  = (y1-y0+1)/52.0
        out.append((cls, cx, cy, w, h))
    return out

# 단일 패턴만 추출
single_idx = np.where(Y.sum(axis=1)==1)[0]
cls_of = {i:int(np.argmax(Y[i])) for i in single_idx}
tr, va = train_test_split(single_idx, test_size=0.2, random_state=42,
                          stratify=[cls_of[i] for i in single_idx])
print('단일 패턴 학습/검증:', len(tr), len(va))

def dump(split, idlist):
    kept = 0
    for i in idlist:
        cls = cls_of[i]
        bxs = boxes_for(X[i], cls)
        if not bxs: continue
        to_rgb(X[i]).save(root/f'images/{split}/wafer_{i}.png')
        with open(root/f'labels/{split}/wafer_{i}.txt','w') as f:
            for c,cx,cy,w,h in bxs:
                f.write(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')
        kept += 1
    return kept
ktr = dump('train', tr); kva = dump('val', va)
print('생성된 라벨 이미지:', ktr, kva)

yaml = f"""path: {root.resolve()}
train: images/train
val: images/val
nc: 8
names: {LABELS}
"""
open(root/'data.yaml','w').write(yaml)
print(yaml)

### 3-1. 생성된 박스 확인 (샘플)

In [ ]:
import matplotlib.patches as patches
sample = list(tr)[:6]
plt.figure(figsize=(13,5))
for j,i in enumerate(sample):
    ax = plt.subplot(2,3,j+1); ax.imshow(to_rgb(X[i])); ax.axis('off')
    for c,cx,cy,w,h in boxes_for(X[i], cls_of[i]):
        x=(cx-w/2)*IMG; y=(cy-h/2)*IMG
        ax.add_patch(patches.Rectangle((x,y),w*IMG,h*IMG,fill=False,color='#22d3ee',lw=2))
        ax.set_title(LABELS[c], fontsize=9)
plt.tight_layout(); plt.show()

## 4. YOLOv8n 학습 (F-02)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')          # 사전학습 가중치에서 파인튜닝
res = model.train(data=str(root/'data.yaml'), epochs=40, imgsz=IMG,
                  batch=32, patience=10, name='wafer_det', verbose=True)
# 검증 지표
metrics = model.val()
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)
best = res.save_dir + '/weights/best.pt'
print('탐지 모델 저장:', best)

## 5. 다중라벨 분류 (Swin Transformer, F-04 근거)
혼합 패턴까지 8종을 동시에 예측. `arr_1`을 그대로 타깃으로 사용(멀티핫).

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import f1_score

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
CSIZE = 224

def to_rgb_arr(a):
    rgb = np.zeros((*a.shape,3), np.uint8)
    rgb[a==1] = (148,163,184); rgb[a==2] = (239,68,68)
    return np.array(Image.fromarray(rgb).resize((CSIZE,CSIZE), Image.NEAREST))

class WaferDS(Dataset):
    def __init__(self, idx): self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, j):
        i = self.idx[j]
        im = to_rgb_arr(X[i]).astype(np.float32)/255.0
        im = (im - 0.5)/0.5
        return torch.tensor(im).permute(2,0,1), torch.tensor(Y[i], dtype=torch.float32)

all_idx = np.arange(len(X))
tr_i, va_i = train_test_split(all_idx, test_size=0.15, random_state=7)
tl = DataLoader(WaferDS(tr_i), batch_size=64, shuffle=True, num_workers=2)
vl = DataLoader(WaferDS(va_i), batch_size=128, shuffle=False, num_workers=2)

net = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=8).to(DEV)
opt = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=0.05)
crit = nn.BCEWithLogitsLoss()

EPOCHS = 8
for ep in range(EPOCHS):
    net.train()
    for xb,yb in tl:
        xb,yb = xb.to(DEV), yb.to(DEV)
        opt.zero_grad(); loss = crit(net(xb), yb); loss.backward(); opt.step()
    # eval
    net.eval(); P=[]; T=[]
    with torch.no_grad():
        for xb,yb in vl:
            p = torch.sigmoid(net(xb.to(DEV))).cpu().numpy()
            P.append(p); T.append(yb.numpy())
    P=np.concatenate(P); T=np.concatenate(T)
    f1 = f1_score(T, (P>0.5).astype(int), average='macro', zero_division=0)
    print(f'epoch {ep+1}/{EPOCHS}  val macro-F1 = {f1:.4f}')

torch.save(net.state_dict(), 'swin_multilabel.pt')
print('분류 모델 저장: swin_multilabel.pt')

## 6. 결과물 정리 → Streamlit 데모로 이동
- `best.pt` (YOLO 탐지) — F-02
- `swin_multilabel.pt` (다중라벨 분류) — F-04 근거
- `label_map.json`

두 파일을 데모 앱과 같은 폴더에 두면 `app.py`가 자동 로드합니다.
분류는 `arr_1`로 학습하므로 Superb 라벨과 독립(Superb 라벨은 플랫폼 연동/피드백 데모용).

In [ ]:
from google.colab import files  # Colab에서 내려받기 (선택)
# files.download('swin_multilabel.pt'); files.download('label_map.json')
print('학습 완료. best.pt / swin_multilabel.pt / label_map.json 준비됨.')